In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
"""
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))
"""

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [ ]:
import os
from PIL import Image
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.cluster import KMeans

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms

from tqdm.notebook import tqdm

In [ ]:
if torch.cuda.is_available():
    device = "cuda:0"
else:
    device = "cpu"
device = torch.device(device)

print(device)

# **Helper Functions**

In [ ]:
def showImages(image_path,label_path,title = "Sample"):
    image = Image.open(image_path).convert("RGB")
    mask = Image.open(mask_path).convert("RGB")
    fig,ax = plt.subplots(1,2,figsize = (12,6))
    ax[0].imshow(image)
    ax[0].set_title(f"{title} Image")
    ax[0].axis('off')
    ax[1].imshow(mask)
    ax[1].set_title(f"{title} Mask")
    ax[1].axis('off')
    plt.show()

In [ ]:
import torch.optim as optim
def train_model(
    model,
    train_loader,
    val_loader,
    test_loader,
    num_epochs,
    device,
    num_classes,
    train_loss,
    train_acc,
    train_prec,
    train_recall,
    train_f1,
    train_iou,
    train_dice,
    val_loss,
    val_acc,
    val_prec,
    val_recall,
    val_f1,
    val_iou,
    val_dice,
    test_loss,
    test_acc,
    test_prec,
    test_recall,
    test_f1,
    test_iou,
    test_dice,
    lr=1e-4,
    save_dir="saved_models",
    monitor_metric="val_iou"
):

    os.makedirs(save_dir, exist_ok=True)
    device = torch.device(device)
    model = model.to(device)
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=lr)

    best_metric_val = -np.inf
    best_epoch = -1
    best_model_path = None

    for epoch in range(num_epochs):
        # ----- TRAIN -----
        model.train()
        epoch_train_loss = 0.0
        epoch_train_acc = 0.0
        epoch_train_prec = 0.0
        epoch_train_rec = 0.0
        epoch_train_f1 = 0.0
        epoch_train_iou = 0.0
        epoch_train_dice = 0.0

        for images, labels in tqdm(train_loader, desc=f"Train Epoch {epoch+1}", leave=False):
            images, labels = images.to(device), labels.to(device)

            optimizer.zero_grad()
            outputs = model(images)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()

            epoch_train_loss += loss.item()

            preds = torch.argmax(outputs, dim=1)
            epoch_train_acc += accuracy_score_batch(preds, labels)
            epoch_train_prec += precision_score_batch(preds, labels, num_classes)
            epoch_train_rec += recall_score_batch(preds, labels, num_classes)
            epoch_train_f1 += f1_score_batch(preds, labels, num_classes)
            epoch_train_iou += iou_score_batch(preds, labels, num_classes)
            epoch_train_dice += dice_score_batch(preds, labels, num_classes)

        train_n = max(1, len(train_loader))
        train_loss.append(epoch_train_loss / train_n)
        train_acc.append(epoch_train_acc / train_n)
        train_prec.append(epoch_train_prec / train_n)
        train_recall.append(epoch_train_rec / train_n)
        train_f1.append(epoch_train_f1 / train_n)
        train_iou.append(epoch_train_iou / train_n)
        train_dice.append(epoch_train_dice / train_n)

        # ----- VALIDATION -----
        model.eval()
        epoch_val_loss = 0.0
        epoch_val_acc = 0.0
        epoch_val_prec = 0.0
        epoch_val_rec = 0.0
        epoch_val_f1 = 0.0
        epoch_val_iou = 0.0
        epoch_val_dice = 0.0

        with torch.no_grad():
            for images, labels in tqdm(val_loader, desc=f"Val Epoch {epoch+1}", leave=False):
                images, labels = images.to(device), labels.to(device)

                outputs = model(images)
                loss = criterion(outputs, labels)
                epoch_val_loss += loss.item()

                preds = torch.argmax(outputs, dim=1)
                epoch_val_acc += accuracy_score_batch(preds, labels)
                epoch_val_prec += precision_score_batch(preds, labels, num_classes)
                epoch_val_rec += recall_score_batch(preds, labels, num_classes)
                epoch_val_f1 += f1_score_batch(preds, labels, num_classes)
                epoch_val_iou += iou_score_batch(preds, labels, num_classes)
                epoch_val_dice += dice_score_batch(preds, labels, num_classes)

        val_n = max(1, len(val_loader))
        val_loss.append(epoch_val_loss / val_n)
        val_acc.append(epoch_val_acc / val_n)
        val_prec.append(epoch_val_prec / val_n)
        val_recall.append(epoch_val_rec / val_n)
        val_f1.append(epoch_val_f1 / val_n)
        val_iou.append(epoch_val_iou / val_n)
        val_dice.append(epoch_val_dice / val_n)

        # ----- TEST -----
        epoch_test_loss = 0.0
        epoch_test_acc = 0.0
        epoch_test_prec = 0.0
        epoch_test_rec = 0.0
        epoch_test_f1 = 0.0
        epoch_test_iou = 0.0
        epoch_test_dice = 0.0

        with torch.no_grad():
            for images, labels in tqdm(test_loader, desc=f"Test Epoch {epoch+1}", leave=False):
                images, labels = images.to(device), labels.to(device)

                outputs = model(images)
                loss = criterion(outputs, labels)
                epoch_test_loss += loss.item()

                preds = torch.argmax(outputs, dim=1)
                epoch_test_acc += accuracy_score_batch(preds, labels)
                epoch_test_prec += precision_score_batch(preds, labels, num_classes)
                epoch_test_rec += recall_score_batch(preds, labels, num_classes)
                epoch_test_f1 += f1_score_batch(preds, labels, num_classes)
                epoch_test_iou += iou_score_batch(preds, labels, num_classes)
                epoch_test_dice += dice_score_batch(preds, labels, num_classes)

        test_n = max(1, len(test_loader))
        test_loss.append(epoch_test_loss / test_n)
        test_acc.append(epoch_test_acc / test_n)
        test_prec.append(epoch_test_prec / test_n)
        test_recall.append(epoch_test_rec / test_n)
        test_f1.append(epoch_test_f1 / test_n)
        test_iou.append(epoch_test_iou / test_n)
        test_dice.append(epoch_test_dice / test_n)

        # ----- Per-epoch checkpoint (save weights & optimizer & metrics) -----
        ckpt = {
            "epoch": epoch + 1,
            "model_state_dict": model.state_dict(),
            "optimizer_state_dict": optimizer.state_dict(),
            # store the metric lists so you have everything in the checkpoint
            "train_loss": train_loss, "train_acc": train_acc, "train_prec": train_prec,
            "train_recall": train_recall, "train_f1": train_f1, "train_iou": train_iou, "train_dice": train_dice,
            "val_loss": val_loss, "val_iou": val_iou, "val_dice": val_dice,
            "val_acc": val_acc, "val_prec": val_prec, "val_recall": val_recall, "val_f1": val_f1,
            "test_loss": test_loss, "test_iou": test_iou, "test_dice": test_dice,
            "test_acc": test_acc, "test_prec": test_prec, "test_recall": test_recall, "test_f1": test_f1,
            "num_classes": num_classes
        }
        epoch_ckpt_path = os.path.join(save_dir, f"model_epoch_{epoch+1}.pth")
        torch.save(ckpt, epoch_ckpt_path)

        # ----- Save best model (based on monitored metric) -----
        metric_value = None
        if monitor_metric == "val_iou":
            metric_value = val_iou[-1] if len(val_iou) > 0 else None
        elif monitor_metric == "val_f1":
            metric_value = val_f1[-1] if len(val_f1) > 0 else None
        elif monitor_metric == "val_loss":
            metric_value = -val_loss[-1] if len(val_loss) > 0 else None
        else:
            metric_value = None

        if metric_value is None or (isinstance(metric_value, float) and np.isnan(metric_value)):
            metric_value = -np.inf

        if metric_value > best_metric_val:
            best_metric_val = metric_value
            best_epoch = epoch + 1
            best_model_path = os.path.join(save_dir, "best_model.pth")
            torch.save(ckpt, best_model_path)

        # ----- PRINT every 2 epochs (or last) -----
        if ((epoch + 1) % 2 == 0) or (epoch == num_epochs - 1):
            print("=" * 90)
            print(f"Epoch [{epoch+1}/{num_epochs}]")
            print(f" Train -> Loss: {train_loss[-1]:.4f} | Acc: {train_acc[-1]:.4f}% | Prec: {train_prec[-1]:.4f}% | Rec: {train_recall[-1]:.4f}% | F1: {train_f1[-1]:.4f}% | IoU: {train_iou[-1]:.4f} | Dice: {train_dice[-1]:.4f}")
            print(f" Val   -> Loss: {val_loss[-1]:.4f} | Acc: {val_acc[-1]:.4f}% | IoU: {val_iou[-1]:.4f} | Dice: {val_dice[-1]:.4f} | Prec: {val_prec[-1]:.4f}% | Rec: {val_recall[-1]:.4f}% | F1: {val_f1[-1]:.4f}%")
            print(f" Test  -> Loss: {test_loss[-1]:.4f} | Acc: {test_acc[-1]:.4f}% | IoU: {test_iou[-1]:.4f} | Dice: {test_dice[-1]:.4f} | Prec: {test_prec[-1]:.4f}% | Rec: {test_recall[-1]:.4f}% | F1: {test_f1[-1]:.4f}%")
            print(f" Best so far -> Epoch: {best_epoch} | {monitor_metric}: {best_metric_val:.6f}")
            print("=" * 90 + "\n")

    return best_model_path

In [ ]:
import torch
import numpy as np

def accuracy_score_batch(preds, labels):
    correct = (preds == labels).sum().item()
    total = labels.numel()
    if total == 0:
        return 0.0
    return (correct / total) * 100.0

def precision_score_batch(preds, labels, num_classes, eps=1e-8):
    preds = preds.view(-1)
    labels = labels.view(-1)
    precisions = []
    for cls in range(num_classes):
        tp = int(((preds == cls) & (labels == cls)).sum().item())
        fp = int(((preds == cls) & (labels != cls)).sum().item())
        denom = tp + fp
        if denom == 0:
            precisions.append(np.nan)
        else:
            precisions.append(tp / (denom + eps))
    return float(np.nanmean(precisions)) * 100.0

def recall_score_batch(preds, labels, num_classes, eps=1e-8):
    preds = preds.view(-1)
    labels = labels.view(-1)
    recalls = []
    for cls in range(num_classes):
        tp = int(((preds == cls) & (labels == cls)).sum().item())
        fn = int(((preds != cls) & (labels == cls)).sum().item())
        denom = tp + fn
        if denom == 0:
            recalls.append(np.nan)
        else:
            recalls.append(tp / (denom + eps))
    return float(np.nanmean(recalls)) * 100.0

def f1_score_batch(preds, labels, num_classes, eps=1e-8):
    preds = preds.view(-1)
    labels = labels.view(-1)
    f1s = []
    for cls in range(num_classes):
        tp = int(((preds == cls) & (labels == cls)).sum().item())
        fp = int(((preds == cls) & (labels != cls)).sum().item())
        fn = int(((preds != cls) & (labels == cls)).sum().item())
        if (tp + fp) == 0 and (tp + fn) == 0:
            f1s.append(np.nan)
            continue
        p = tp / (tp + fp + eps) if (tp + fp) > 0 else 0.0
        r = tp / (tp + fn + eps) if (tp + fn) > 0 else 0.0
        if (p + r) == 0:
            f1s.append(np.nan)
        else:
            f1s.append((2 * p * r) / (p + r + eps))
    return float(np.nanmean(f1s)) * 100.0

def iou_score_batch(preds, labels, num_classes):
    preds = preds.view(-1)
    labels = labels.view(-1)
    ious = []
    for cls in range(num_classes):
        pred_inds = preds == cls
        target_inds = labels == cls
        intersection = int((pred_inds & target_inds).sum().item())
        union = int((pred_inds | target_inds).sum().item())
        if union == 0:
            ious.append(np.nan)
        else:
            ious.append(intersection / union)
    return float(np.nanmean(ious))

def dice_score_batch(preds, labels, num_classes, eps=1e-8):
    preds = preds.view(-1)
    labels = labels.view(-1)
    dices = []
    for cls in range(num_classes):
        pred_inds = preds == cls
        target_inds = labels == cls
        intersection = int((pred_inds & target_inds).sum().item())
        denom = int(pred_inds.sum().item() + target_inds.sum().item())
        if denom == 0:
            dices.append(np.nan)
        else:
            dices.append((2.0 * intersection) / (denom + eps))
    return float(np.nanmean(dices))

In [ ]:
import matplotlib.pyplot as plt
import torch

def show_image_gt_pred(dataset, model, idx=0, device="cuda"):
    model.eval()
    
    image, mask = dataset[idx]
    orig_img = image.permute(1, 2, 0).cpu().numpy()
    
    image = image.unsqueeze(0).to(device)
    mask = mask.cpu().numpy()
    
    with torch.no_grad():
        output = model(image)
        pred = torch.argmax(output, dim=1).squeeze(0).cpu().numpy()
    
    plt.figure(figsize=(20, 5))
    
    plt.subplot(1, 4, 1)
    plt.imshow(orig_img)
    plt.title("Original Image")
    plt.axis("off")
    
    plt.subplot(1, 4, 2)
    plt.imshow(mask, cmap="gray")
    plt.title("Ground Truth Mask")
    plt.axis("off")
    
    plt.subplot(1, 4, 3)
    plt.imshow(pred, cmap="gray")
    plt.title("Predicted Mask")
    plt.axis("off")

    plt.subplot(1, 4, 4)
    plt.imshow(orig_img)
    plt.imshow(pred, cmap="jet", alpha=0.5)  # using 'jet' for better visibility
    plt.title("Overlay (Image + Prediction)")
    plt.axis("off")
    
    plt.show()

In [ ]:
def load_checkpoint(model, ckpt_path, optimizer=None, device="cuda"):
    """
    Loads a checkpoint (saved dict) into model (and optimizer if provided).
    Returns the checkpoint dict (contains epoch, metric-lists, etc.).
    Usage:
      ckpt = load_checkpoint(model, "saved_models/model_epoch_10.pth", optimizer=None, device="cpu")
      model.eval(); # then run inference
    """
    ckpt = torch.load(ckpt_path, map_location=device)
    model.load_state_dict(ckpt["model_state_dict"])
    if optimizer is not None and "optimizer_state_dict" in ckpt:
        optimizer.load_state_dict(ckpt["optimizer_state_dict"])
    return ckpt

In [ ]:
# Fixed measure_dataloader (reset iterator after warmup)
import time, numpy as np, torch

def measure_dataloader(model, loader, device=None, warmup_batches=10, n_batches=None,
                       max_total_batches=1000, verbose=True):
    """
    Fixed version: resets iterator after warmup so measurement starts from beginning.
    Returns: numpy array of per-sample times (seconds).
    """
    # normalize device (simple)
    if isinstance(device, str):
        device = torch.device(device)
    elif device is None:
        device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    else:
        try:
            device = torch.device(str(device))
        except Exception:
            device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    # Basic checks
    if loader is None:
        raise ValueError("loader is None. Provide a valid DataLoader.")
    try:
        loader_len = len(loader)
    except Exception:
        loader_len = None

    model = model.to(device)
    model.eval()

    times = []
    total_measured_batches = 0
    total_seen_batches = 0

    it = iter(loader)

    # Warmup: run some batches (wrap if loader shorter than warmup)
    with torch.inference_mode():
        for _ in range(warmup_batches):
            try:
                batch = next(it)
            except StopIteration:
                it = iter(loader)
                try:
                    batch = next(it)
                except StopIteration:
                    raise ValueError("DataLoader yielded no batches during warmup. Check dataset/transforms.")
            imgs = batch[0].to(device)
            _ = model(imgs)
        if device.type.startswith("cuda"):
            torch.cuda.synchronize()

    # IMPORTANT FIX: reset iterator so timed measurements start fresh
    it = iter(loader)

    # Timed measurement loop
    with torch.inference_mode():
        while True:
            if n_batches is not None and total_measured_batches >= n_batches:
                break
            if total_seen_batches >= max_total_batches:
                if verbose:
                    print(f"[measure_dataloader] reached max_total_batches={max_total_batches}, stopping.")
                break

            try:
                batch = next(it)
            except StopIteration:
                # iterator finished
                break

            imgs = batch[0].to(device)
            bs = imgs.size(0)
            if bs == 0:
                total_seen_batches += 1
                continue

            start = time.perf_counter()
            _ = model(imgs)
            if device.type.startswith("cuda"):
                torch.cuda.synchronize()
            end = time.perf_counter()

            batch_time = end - start
            times.append((batch_time, bs))
            total_measured_batches += 1
            total_seen_batches += 1

    if len(times) == 0:
        raise ValueError(
            "No batches were measured. Possible causes:\n"
            " - val_loader is empty or yields zero-sized batches,\n"
            " - warmup consumed all data and iterator exhausted before measurements (fixed by reset),\n"
            " - n_batches was set to 0.\n"
            "Check your DataLoader and try setting n_batches=1 for a quick test."
        )

    # Expand to per-sample times
    per_sample_times_list = [np.full(bs, (t / bs), dtype=float) for (t, bs) in times]
    per_sample_times = np.concatenate(per_sample_times_list).astype(float)

    if verbose:
        print(f"\n[measure_dataloader] device={device}, batches_measured={len(times)}, total_samples_measured={per_sample_times.size}")
        if loader_len is not None:
            print(f"[measure_dataloader] reported loader_len={loader_len}")
        print(f"[measure_dataloader] warmup_batches={warmup_batches}, requested_n_batches={n_batches}")

    return per_sample_times

# **Data Loading and Augmentation**

In [ ]:
test_path = "/kaggle/input/whu-building-dataset/WHU/test/Image"
test_labels_path = "/kaggle/input/whu-building-dataset/WHU/test/Mask"
train_path = "/kaggle/input/whu-building-dataset/WHU/train/Image"
train_labels_path = "/kaggle/input/whu-building-dataset/WHU/train/Mask"
val_path = "/kaggle/input/whu-building-dataset/WHU/val/Image"
val_labels_path = "/kaggle/input/whu-building-dataset/WHU/val/Mask"

In [ ]:
test = os.listdir(test_path)
test_label = os.listdir(test_labels_path)
train = os.listdir(train_path)
train_label = os.listdir(train_labels_path)
val = os.listdir(val_path)
val_label = os.listdir(val_labels_path)

In [ ]:
print(f"Total test images : {len(test)}")
print(f"Total val images : {len(val)}")
print(f"Total train images : {len(train)}")

In [ ]:
img_path = "/kaggle/input/whu-building-dataset/WHU/test/Image/test_0002.png"
mask_path = "/kaggle/input/whu-building-dataset/WHU/test/Mask/test_0002.png"
showImages(img_path,mask_path)

In [ ]:
COLOR_MAP ={
    (0,0,0):0, #surrounding
    (255,255,255):1 #buildings
}

In [ ]:
class BuildingDataset(Dataset):
    def __init__(self, image_path, label_path, transform=None):
        self.image_path = image_path
        self.label_path = label_path
        self.transform = transform
        self.imgs = os.listdir(image_path)

        # label transform (NEAREST for segmentation masks!)
        self.label_transform = transforms.Compose([
            transforms.Resize((224, 224), interpolation=InterpolationMode.NEAREST),
            transforms.PILToTensor()
        ])

    def __len__(self):
        return len(self.imgs)

    def __getitem__(self, idx):
        img_name = self.imgs[idx]
        image = Image.open(os.path.join(self.image_path, img_name)).convert("RGB")
        label = Image.open(os.path.join(self.label_path, img_name)).convert("RGB")
        label = self.rgb_to_label(np.array(label))
        label = Image.fromarray(label.astype(np.uint8))

        if self.transform:
            image = self.transform(image)
        label = self.label_transform(label).squeeze(0)   # (H, W)

        return image, label.long()

    def rgb_to_label(self, label):
        label_mask = np.zeros(label.shape[:2], dtype=np.uint8)
        for color, value in COLOR_MAP.items():
            label_mask[np.all(label == color, axis=-1)] = value
        return label_mask

    def __get_original_image__(self, idx):
        img_name = self.image_filenames[idx]
        image_path = os.path.join(self.image_dir, img_name)
        image = Image.open(image_path).convert("RGB")
        return image

In [ ]:
IMG_SIZE = (224, 224)

train_transform = transforms.Compose([
    transforms.Resize(IMG_SIZE),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485,0.456,0.406], std=[0.229,0.224,0.225]),
])

val_transform = transforms.Compose([
    transforms.Resize(IMG_SIZE),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485,0.456,0.406], std=[0.229,0.224,0.225]),
])

In [ ]:
from torchvision.transforms import InterpolationMode

train_dataset = BuildingDataset(train_path, train_labels_path, transform=train_transform)
val_dataset   = BuildingDataset(val_path,   val_labels_path,   transform=val_transform)
test_dataset  = BuildingDataset(test_path,  test_labels_path,  transform=val_transform)

train_loader = DataLoader(train_dataset, batch_size=4, shuffle=True)
val_loader   = DataLoader(val_dataset,   batch_size=4, shuffle=False)
test_loader  = DataLoader(test_dataset,  batch_size=4, shuffle=False)

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"
num_classes = len(COLOR_MAP)

# **Model**

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

# -------------------------
# Basic ConvBlock (fixed)
# -------------------------
class ConvBlock(nn.Module):
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.conv1 = nn.Conv2d(in_ch, out_ch, kernel_size=3, padding=1, bias=False)
        self.bn1 = nn.BatchNorm2d(out_ch)
        self.relu = nn.ReLU(inplace=True)
        self.conv2 = nn.Conv2d(out_ch, out_ch, kernel_size=3, padding=1, bias=False)
        self.bn2 = nn.BatchNorm2d(out_ch)

    def forward(self, x):
        x = self.conv1(x)
        x = self.bn1(x)
        x = self.relu(x)
        x = self.conv2(x)
        x = self.bn2(x)
        x = self.relu(x)
        return x

# -------------------------
# Choose MambaBlock implementation:
# - use real mamba_ssm.Mamba only if import succeeds AND CUDA is available
# - otherwise use convolutional fallback
# -------------------------
_use_real_mamba = False
try:
    import mamba_ssm
    # ensure CUDA is available, because the real mamba uses CUDA kernels
    if torch.cuda.is_available():
        from mamba_ssm import Mamba
        _use_real_mamba = True
    else:
        print("mamba_ssm found but CUDA is not available. Falling back to convolutional MambaBlock (CPU-safe).")
except Exception:
    # not installed or other import error
    _use_real_mamba = False

if _use_real_mamba:
    print("Using real mamba_ssm.Mamba (GPU).")

    class MambaBlock(nn.Module):
        """
        Wraps the Mamba module (requires CUDA).
        Projects channel dimension in/out so it can be used inside a conv U-Net.
        """
        def __init__(self, in_c, d_model, d_state=16, d_conv=4, expand=2):
            super().__init__()
            self.proj = nn.Linear(in_c, d_model)
            self.mamba = Mamba(d_model, d_state, d_conv, expand)
            self.norm = nn.LayerNorm(d_model)
            self.proj_out = nn.Linear(d_model, in_c)

        def forward(self, x):
            # x: (B, C, H, W)
            b, c, h, w = x.shape
            # flatten spatial dims
            x_flat = x.permute(0, 2, 3, 1).contiguous().view(b, h * w, c)
            # IMPORTANT: Mamba expects CUDA tensors; ensure x_flat is on cuda
            x_flat = x_flat.to(next(self.mamba.parameters()).device)
            x_m = self.norm(self.proj(x_flat))
            x_m = self.mamba(x_m)
            x_m = self.proj_out(x_m)
            x_out = x_m.view(b, h, w, c).permute(0, 3, 1, 2).contiguous()
            return x_out + x

else:
    print("Using convolutional fallback MambaBlock (CPU/GPU safe).")

    class MambaBlock(nn.Module):
        """
        Lightweight convolutional approximation of the Mamba block.
        """
        def __init__(self, in_c, d_model, d_state=16, d_conv=3, expand=2):
            super().__init__()
            inner_dim = int(d_model * expand)
            self.in_proj = nn.Conv2d(in_c, inner_dim * 2, kernel_size=1, bias=False)
            self.conv_spatial = nn.Conv2d(inner_dim, inner_dim, kernel_size=d_conv, padding=d_conv//2, groups=inner_dim, bias=False)
            self.act = nn.SiLU()
            self.proj_out = nn.Conv2d(inner_dim, in_c, kernel_size=1, bias=False)

        def forward(self, x):
            residual = x
            x_in = self.in_proj(x)
            gate, conv_part = x_in.chunk(2, dim=1)
            conv_out = self.conv_spatial(conv_part)
            x_out = conv_out * torch.sigmoid(gate)
            x_out = self.proj_out(self.act(x_out))
            return x_out + residual

# -------------------------
# Full MambaUNet (U-Net style)
# -------------------------
class MambaUNet(nn.Module):
    def __init__(self, in_channels=3, out_channels=2, base_channels=64):
        super().__init__()
        c = base_channels
        # Encoder
        self.conv1 = ConvBlock(in_channels, c)
        self.mamba1 = MambaBlock(c, d_model=c)
        self.pool1 = nn.MaxPool2d(2)

        self.conv2 = ConvBlock(c, c*2)
        self.mamba2 = MambaBlock(c*2, d_model=c*2)
        self.pool2 = nn.MaxPool2d(2)

        self.conv3 = ConvBlock(c*2, c*4)
        self.mamba3 = MambaBlock(c*4, d_model=c*4)
        self.pool3 = nn.MaxPool2d(2)

        self.conv4 = ConvBlock(c*4, c*8)
        self.mamba4 = MambaBlock(c*8, d_model=c*8)
        self.pool4 = nn.MaxPool2d(2)

        # Bottleneck
        self.conv_bottleneck = ConvBlock(c*8, c*16)
        self.mamba_bottleneck = MambaBlock(c*16, d_model=c*16)

        # Decoder
        self.up4 = nn.ConvTranspose2d(c*16, c*8, kernel_size=2, stride=2)
        self.conv5 = ConvBlock(c*16, c*8)

        self.up3 = nn.ConvTranspose2d(c*8, c*4, kernel_size=2, stride=2)
        self.conv6 = ConvBlock(c*8, c*4)

        self.up2 = nn.ConvTranspose2d(c*4, c*2, kernel_size=2, stride=2)
        self.conv7 = ConvBlock(c*4, c*2)

        self.up1 = nn.ConvTranspose2d(c*2, c, kernel_size=2, stride=2)
        self.conv8 = ConvBlock(c*2, c)

        # Final conv
        self.conv_out = nn.Conv2d(c, out_channels, kernel_size=1)

    def forward(self, x):
        # Encoder
        c1 = self.conv1(x)
        c1 = self.mamba1(c1)
        p1 = self.pool1(c1)

        c2 = self.conv2(p1)
        c2 = self.mamba2(c2)
        p2 = self.pool2(c2)

        c3 = self.conv3(p2)
        c3 = self.mamba3(c3)
        p3 = self.pool3(c3)

        c4 = self.conv4(p3)
        c4 = self.mamba4(c4)
        p4 = self.pool4(c4)

        # Bottleneck
        cb = self.conv_bottleneck(p4)
        cb = self.mamba_bottleneck(cb)

        # Decoder
        u4 = self.up4(cb)
        if u4.shape[2:] != c4.shape[2:]:
            u4 = F.interpolate(u4, size=c4.shape[2:], mode='bilinear', align_corners=False)
        u4 = torch.cat([u4, c4], dim=1)
        c5 = self.conv5(u4)

        u3 = self.up3(c5)
        if u3.shape[2:] != c3.shape[2:]:
            u3 = F.interpolate(u3, size=c3.shape[2:], mode='bilinear', align_corners=False)
        u3 = torch.cat([u3, c3], dim=1)
        c6 = self.conv6(u3)

        u2 = self.up2(c6)
        if u2.shape[2:] != c2.shape[2:]:
            u2 = F.interpolate(u2, size=c2.shape[2:], mode='bilinear', align_corners=False)
        u2 = torch.cat([u2, c2], dim=1)
        c7 = self.conv7(u2)

        u1 = self.up1(c7)
        if u1.shape[2:] != c1.shape[2:]:
            u1 = F.interpolate(u1, size=c1.shape[2:], mode='bilinear', align_corners=False)
        u1 = torch.cat([u1, c1], dim=1)
        c8 = self.conv8(u1)

        out = self.conv_out(c8)
        return out

# -------------------------
# Smoke test (auto device selection)
# -------------------------
if __name__ == "__main__":
    device = "cuda" if torch.cuda.is_available() else "cpu"
    print("Selected device:", device)
    
    # Initialize model
    model = MambaUNet(in_channels=3, out_channels=2, base_channels=32).to(device)
    
    # Print number of trainable parameters
    num_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f"Number of trainable parameters: {num_params:,}")
    
    # Smoke test with a dummy input
    x = torch.randn(4, 3, 224, 224).to(device)
    y = model(x)
    print("Output shape:", y.shape)  # should be (4, out_channels, 224, 224)

In [ ]:
model2 = MambaUNet().to(device)
print(f"Model initialized on {device}")

# **Training and Testing Loop**

In [ ]:
train_loss2 = []
train_acc2 = []
train_prec2 = []
train_recall2 = []
train_f12 = []
train_iou2 = []    # NEW: store train IoU per epoch
train_dice2= []   # NEW: store train Dice per epoch

val_loss2 = []
val_iou2 = []
val_dice2 = []
val_acc2 = []
val_prec2 = []
val_recall2 = []
val_f12 = []

test_loss2 = []
test_iou2 = []
test_dice2 = []
test_acc2 = []
test_prec2 = []
test_recall2 = []
test_f12 = []

In [ ]:
train_loss3 = []
train_acc3 = []
train_prec3 = []
train_recall3 = []
train_f13 = []
train_iou3 = []    # NEW: store train IoU per epoch
train_dice3= []   # NEW: store train Dice per epoch

val_loss3 = []
val_iou3 = []
val_dice3 = []
val_acc3 = []
val_prec3 = []
val_recall3 = []
val_f13 = []

test_loss3 = []
test_iou3 = []
test_dice3 = []
test_acc3 = []
test_prec3 = []
test_recall3 = []
test_f13 = []

In [ ]:
train_loss4 = []
train_acc4 = []
train_prec4 = []
train_recall4 = []
train_f14 = []
train_iou4 = []    # NEW: store train IoU per epoch
train_dice4= []   # NEW: store train Dice per epoch

val_loss4 = []
val_iou4 = []
val_dice4 = []
val_acc4 = []
val_prec4 = []
val_recall4 = []
val_f14 = []

test_loss4 = []
test_iou4 = []
test_dice4 = []
test_acc4 = []
test_prec4 = []
test_recall4 = []
test_f14 = []

In [ ]:
train_model(
    model2,
    train_loader,
    val_loader,
    test_loader,
    num_epochs=10,
    device=device,
    num_classes=num_classes,
    train_loss=train_loss2,
    train_acc=train_acc2,
    train_prec=train_prec2,
    train_recall=train_recall2,
    train_f1=train_f12,
    train_iou=train_iou2,
    train_dice=train_dice2,
    val_loss=val_loss2,
    val_acc=val_acc2,
    val_prec=val_prec2,
    val_recall=val_recall2,
    val_f1=val_f12,
    val_iou=val_iou2,
    val_dice=val_dice2,
    test_loss=test_loss2,
    test_acc=test_acc2,
    test_prec=test_prec2,
    test_recall=test_recall2,
    test_f1=test_f12,
    test_iou=test_iou2,
    test_dice=test_dice2,
)

In [ ]:
print(f" Train -> Loss: {train_loss2[-1]:.4f} | Acc: {train_acc2[-1]:.4f}% | Prec: {train_prec2[-1]:.4f}% | Rec: {train_recall2[-1]:.4f}% | F1: {train_f12[-1]:.4f}% | IoU: {train_iou2[-1]:.4f} | Dice: {train_dice2[-1]:.4f}")
print(f" Val   -> Loss: {val_loss2[-1]:.4f} | Acc: {val_acc2[-1]:.4f}% | IoU: {val_iou2[-1]:.4f} | Dice: {val_dice2[-1]:.4f} | Prec: {val_prec2[-1]:.4f}% | Rec: {val_recall2[-1]:.4f}% | F1: {val_f12[-1]:.4f}%")
print(f" Test  -> Loss: {test_loss2[-1]:.4f} | Acc: {test_acc2[-1]:.4f}% | IoU: {test_iou2[-1]:.4f} | Dice: {test_dice2[-1]:.4f} | Prec: {test_prec2[-1]:.4f}% | Rec: {test_recall2[-1]:.4f}% | F1: {test_f12[-1]:.4f}%")
print("=" * 90 + "\n")

In [ ]:
train_model(
    model2,
    train_loader,
    val_loader,
    test_loader,
    num_epochs=2,
    device=device,
    num_classes=num_classes,
    train_loss=train_loss3,
    train_acc=train_acc3,
    train_prec=train_prec3,
    train_recall=train_recall3,
    train_f1=train_f13,
    train_iou=train_iou3,
    train_dice=train_dice3,
    val_loss=val_loss3,
    val_acc=val_acc3,
    val_prec=val_prec3,
    val_recall=val_recall3,
    val_f1=val_f13,
    val_iou=val_iou3,
    val_dice=val_dice3,
    test_loss=test_loss3,
    test_acc=test_acc3,
    test_prec=test_prec3,
    test_recall=test_recall3,
    test_f1=test_f13,
    test_iou=test_iou3,
    test_dice=test_dice3,
)

In [ ]:
# Assuming the _2 and _3 lists have been populated with values,
# this code concatenates them into new lists.

# Aggregated Training Metrics
train_loss = train_loss2 + train_loss3
train_acc = train_acc2 + train_acc3
train_prec = train_prec2 + train_prec3
train_recall = train_recall2 + train_recall3
train_f1 = train_f12 + train_f13
train_iou = train_iou2 + train_iou3
train_dice = train_dice2 + train_dice3

# ---

# Aggregated Validation Metrics
val_loss = val_loss2 + val_loss3
val_iou = val_iou2 + val_iou3
val_dice = val_dice2 + val_dice3
val_acc = val_acc2 + val_acc3
val_prec = val_prec2 + val_prec3
val_recall = val_recall2 + val_recall3
val_f1 = val_f12 + val_f13

# ---

# Aggregated Test Metrics
test_loss = test_loss2 + test_loss3
test_iou = test_iou2 + test_iou3
test_dice = test_dice2 + test_dice3
test_acc = test_acc2 + test_acc3
test_prec = test_prec2 + test_prec3
test_recall = test_recall2 + test_recall3
test_f1 = test_f12 + test_f13

In [ ]:
train_model(
    model2,
    train_loader,
    val_loader,
    test_loader,
    num_epochs=10,
    device=device,
    num_classes=num_classes,
    train_loss=train_loss3,
    train_acc=train_acc3,
    train_prec=train_prec3,
    train_recall=train_recall3,
    train_f1=train_f13,
    train_iou=train_iou3,
    train_dice=train_dice3,
    val_loss=val_loss3,
    val_acc=val_acc3,
    val_prec=val_prec3,
    val_recall=val_recall3,
    val_f1=val_f13,
    val_iou=val_iou3,
    val_dice=val_dice3,
    test_loss=test_loss3,
    test_acc=test_acc3,
    test_prec=test_prec3,
    test_recall=test_recall3,
    test_f1=test_f13,
    test_iou=test_iou3,
    test_dice=test_dice3,
)

In [ ]:
# Assuming the _2 and _3 lists have been populated with values,
# this code concatenates them into new lists.

# Aggregated Training Metrics
train_loss = train_loss + train_loss3
train_acc = train_acc + train_acc3
train_prec = train_prec + train_prec3
train_recall = train_recall + train_recall3
train_f1 = train_f1 + train_f13
train_iou = train_iou + train_iou3
train_dice = train_dice + train_dice3

# ---

# Aggregated Validation Metrics
val_loss = val_loss + val_loss3
val_iou = val_iou + val_iou3
val_dice = val_dice + val_dice3
val_acc = val_acc + val_acc3
val_prec = val_prec + val_prec3
val_recall = val_recall + val_recall3
val_f1 = val_f1 + val_f13

# ---

# Aggregated Test Metrics
test_loss = test_loss + test_loss3
test_iou = test_iou + test_iou3
test_dice = test_dice + test_dice3
test_acc = test_acc + test_acc3
test_prec = test_prec + test_prec3
test_recall = test_recall + test_recall3
test_f1 = test_f1 + test_f13

In [ ]:
train_model(
    model2,
    train_loader,
    val_loader,
    test_loader,
    num_epochs=8,
    device=device,
    num_classes=num_classes,
    train_loss=train_loss3,
    train_acc=train_acc3,
    train_prec=train_prec3,
    train_recall=train_recall3,
    train_f1=train_f13,
    train_iou=train_iou3,
    train_dice=train_dice3,
    val_loss=val_loss3,
    val_acc=val_acc3,
    val_prec=val_prec3,
    val_recall=val_recall3,
    val_f1=val_f13,
    val_iou=val_iou3,
    val_dice=val_dice3,
    test_loss=test_loss3,
    test_acc=test_acc3,
    test_prec=test_prec3,
    test_recall=test_recall3,
    test_f1=test_f13,
    test_iou=test_iou3,
    test_dice=test_dice3,
)

In [ ]:
# Assuming the _2 and _3 lists have been populated with values,
# this code concatenates them into new lists.

# Aggregated Training Metrics
train_loss = train_loss + train_loss3
train_acc = train_acc + train_acc3
train_prec = train_prec + train_prec3
train_recall = train_recall + train_recall3
train_f1 = train_f1 + train_f13
train_iou = train_iou + train_iou3
train_dice = train_dice + train_dice3

# ---

# Aggregated Validation Metrics
val_loss = val_loss + val_loss3
val_iou = val_iou + val_iou3
val_dice = val_dice + val_dice3
val_acc = val_acc + val_acc3
val_prec = val_prec + val_prec3
val_recall = val_recall + val_recall3
val_f1 = val_f1 + val_f13

# ---

# Aggregated Test Metrics
test_loss = test_loss + test_loss3
test_iou = test_iou + test_iou3
test_dice = test_dice + test_dice3
test_acc = test_acc + test_acc3
test_prec = test_prec + test_prec3
test_recall = test_recall + test_recall3
test_f1 = test_f1 + test_f13

In [ ]:
test_acc

In [ ]:
train_model(
    model2,
    train_loader,
    val_loader,
    test_loader,
    num_epochs=2,
    device=device,
    num_classes=num_classes,
    train_loss=train_loss3,
    train_acc=train_acc3,
    train_prec=train_prec3,
    train_recall=train_recall3,
    train_f1=train_f13,
    train_iou=train_iou3,
    train_dice=train_dice3,
    val_loss=val_loss3,
    val_acc=val_acc3,
    val_prec=val_prec3,
    val_recall=val_recall3,
    val_f1=val_f13,
    val_iou=val_iou3,
    val_dice=val_dice3,
    test_loss=test_loss3,
    test_acc=test_acc3,
    test_prec=test_prec3,
    test_recall=test_recall3,
    test_f1=test_f13,
    test_iou=test_iou3,
    test_dice=test_dice3,
)

In [ ]:
# Assuming the _2 and _3 lists have been populated with values,
# this code concatenates them into new lists.

# Aggregated Training Metrics
train_loss = train_loss + train_loss3
train_acc = train_acc + train_acc3
train_prec = train_prec + train_prec3
train_recall = train_recall + train_recall3
train_f1 = train_f1 + train_f13
train_iou = train_iou + train_iou3
train_dice = train_dice + train_dice3

# ---

# Aggregated Validation Metrics
val_loss = val_loss + val_loss3
val_iou = val_iou + val_iou3
val_dice = val_dice + val_dice3
val_acc = val_acc + val_acc3
val_prec = val_prec + val_prec3
val_recall = val_recall + val_recall3
val_f1 = val_f1 + val_f13

# ---

# Aggregated Test Metrics
test_loss = test_loss + test_loss3
test_iou = test_iou + test_iou3
test_dice = test_dice + test_dice3
test_acc = test_acc + test_acc3
test_prec = test_prec + test_prec3
test_recall = test_recall + test_recall3
test_f1 = test_f1 + test_f13

In [ ]:
test_acc

In [ ]:
# Assuming the aggregated lists (train_loss, val_loss, etc.) have been populated
# by concatenating the _2 and _3 runs (e.g., train_loss = train_loss2 + train_loss3).

# ----------------------------------------------------------------------
# Aggregated Training Metrics - Unique Values
# ----------------------------------------------------------------------
unique_train_loss = list(set(train_loss))
unique_train_acc = list(set(train_acc))
unique_train_prec = list(set(train_prec))
unique_train_recall = list(set(train_recall))
unique_train_f1 = list(set(train_f1))
unique_train_iou = list(set(train_iou))
unique_train_dice = list(set(train_dice))

# ----------------------------------------------------------------------
# Aggregated Validation Metrics - Unique Values
# ----------------------------------------------------------------------
unique_val_loss = list(set(val_loss))
unique_val_iou = list(set(val_iou))
unique_val_dice = list(set(val_dice))
unique_val_acc = list(set(val_acc))
unique_val_prec = list(set(val_prec))
unique_val_recall = list(set(val_recall))
unique_val_f1 = list(set(val_f1))

# ----------------------------------------------------------------------
# Aggregated Test Metrics - Unique Values
# ----------------------------------------------------------------------
unique_test_loss = list(set(test_loss))
unique_test_iou = list(set(test_iou))
unique_test_dice = list(set(test_dice))
unique_test_acc = list(set(test_acc))
unique_test_prec = list(set(test_prec))
unique_test_recall = list(set(test_recall))
unique_test_f1 = list(set(test_f1))

In [ ]:
len(unique_test_loss)

In [ ]:
unique_test_acc[-1]

In [ ]:
plt.figure(figsize=(28,5))

# Loss plot
plt.subplot(1,2,1)
plt.plot(unique_train_loss, label='Train Loss')
plt.plot(unique_val_loss, label='Validation Loss')
plt.plot(unique_test_loss, label='Test Loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.title('Loss over Epochs')
plt.legend()
plt.grid(True)

plt.show()

In [ ]:
plt.figure(figsize=(28,5))

# IoU plot
plt.subplot(1,2,1)
plt.plot(unique_train_iou, label='Training IoU', marker='o')
plt.plot(unique_val_iou, label='Validation IoU', marker='o')
plt.plot(unique_test_iou, label='Test IoU', marker='o')
plt.xlabel('Epoch')
plt.ylabel('IoU')
plt.title('IoU over Epochs')
plt.legend()
plt.grid(True)

plt.show()

In [ ]:
plt.figure(figsize=(28,5))
# Accuracy plot
plt.subplot(1,2,2)
plt.plot(unique_train_acc, label='Train Acc')
plt.plot(unique_val_acc, label='Validation Acc')
plt.plot(unique_test_acc, label='Test Acc')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.title('Accuracy over Epochs')
plt.legend()
plt.grid(True)
plt.show()

In [ ]:
plt.figure(figsize=(28,5))
# Dice plot
plt.subplot(1,2,2)
plt.plot(unique_train_dice, label='Training Dice', marker='o')
plt.plot(unique_val_dice, label='Validation Dice', marker='o')
plt.plot(unique_test_dice, label='Test Dice', marker='o')
plt.xlabel('Epoch')
plt.ylabel('Dice Score')
plt.title('Dice Score over Epochs')
plt.legend()
plt.grid(True)
plt.show()

In [ ]:
show_image_gt_pred(test_dataset, model2, idx=1, device=device)
show_image_gt_pred(test_dataset, model2, idx=2, device=device)
show_image_gt_pred(test_dataset, model2, idx=3, device=device)
show_image_gt_pred(test_dataset, model2, idx=4, device=device)
show_image_gt_pred(test_dataset, model2, idx=5, device=device)

In [ ]:
ckpt = load_checkpoint(model2, "saved_models/best_model.pth", device="cuda")
model2.eval()

In [ ]:
from torchinfo import summary
summary(model, input_size=(4, 3, 224, 224))

In [ ]:
!pip install fvcore
from fvcore.nn import FlopCountAnalysis

input_tensor = torch.randn(1, 3, 224, 224).to(device)
flops = FlopCountAnalysis(model, input_tensor)
print("FLOPs:", flops.total())
print("FLOPs (in GigaFLOPs):", flops.total() / 1e9)

In [ ]:
overall_accuracy = unique_test_acc[-1]  # last epoch or best one
print("Overall Accuracy (OA):", overall_accuracy)

In [ ]:
import numpy as np

best_epoch = np.argmax(unique_val_prec)  # or val_dice
print(f"Best Epoch: {best_epoch+1}")
print(f"Val Accuracy: {unique_val_acc[best_epoch]:.4f}")
print(f"Val IoU: {unique_val_iou[best_epoch]:.4f}")
print(f"Val Dice: {unique_val_dice[best_epoch]:.4f}")
print(f"Val Precession: {unique_val_prec[best_epoch]:.4f}")

In [ ]:
import numpy as np

best_epoch = np.argmax(unique_val_iou)  # or val_dice
print(f"Best Epoch: {best_epoch+1}")
print(f"Val Accuracy: {unique_val_acc[best_epoch]:.4f}")
print(f"Val IoU: {unique_val_iou[best_epoch]:.4f}")
print(f"Val Dice: {unique_val_dice[best_epoch]:.4f}")